# Study on Plan pattern of agentic programming

## Prepare

In [8]:
import json
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print(f'Import LangChain V{langchain.__version__}')

Import LangChain V1.2.7


In [7]:
with open('api-key/deepseek.txt', 'r') as f:
    llm = ChatOpenAI(
        base_url="https://api.deepseek.com",
        api_key=f.read().strip(),
        model="deepseek-v4-flash",
        temperature=0,
    )

# print(
#     StrOutputParser().invoke(
#         llm.invoke([("human", "Introduce yourself briefly")])
#     )
# )

## Definition of agents

Define plan, write and synthesis agents

In [9]:
plan_agent = (
    ChatPromptTemplate.from_messages([
        ("system", """你是一名资深技术写手和内容策略师，在这个任务中：
         - 根据用户给出的主题制定摘要的要点计划；
         - 将摘要分拆为多个要点，并给出每个要点的编写规则，如方向、思路、重点、字数等；
         - 所有要点要给定统一的写作风格；
         - 最终输出所有要点编写提示语的列表，json列表格式。"""),
        ("user", "{topic}")
    ]) | llm | StrOutputParser()
)

In [11]:
point_agent = (
    ChatPromptTemplate.from_messages([
        ("system", "作为一名资深技术写手，根据提示编写一个摘要要点"),
        ("user", "{key_prompt}")
    ]) | llm | StrOutputParser()
)

In [ ]:
syn_agent = (
    ChatPromptTemplate.from_messages([
        ("system", """作为一名资深技术写手，根据主题和所有的要点，整合输出约 200 字的摘要。
         Markdown 格式，具体为：
         ### 摘要

         摘要主旨

         - 分3到5个条目
         """),
        ("user", """主题：{topic}
         要点：{points}""")
    ]) | llm | StrOutputParser()
)

## Process

In [13]:
topic = "强化学习在 AI 中的重要性"

In [14]:
plan = plan_agent.invoke({"topic": topic})
prompts = json.loads(plan)
print(f"Generate {len(prompts)} point prompt:")
for prompt in prompts:
    print(f"- {prompt}")

Generate 5 point prompt:
- {'id': 1, 'title': '强化学习的定义与核心价值', 'direction': '概念引入', 'thinking': '从强化学习的基本定义（智能体通过与环境交互、根据奖惩信号学习最优策略）出发，强调其与监督/无监督学习的本质区别：无需预先标注数据，而是通过试错探索自主进化。', 'focus': '突出强化学习在解决序列决策问题上的独特性，以及它如何模拟生物学习过程', 'word_count': 150, 'style': '专业严谨、逻辑清晰，适当使用“试错”“奖惩”等直观比喻'}
- {'id': 2, 'title': '强化学习与其他学习范式的对比', 'direction': '对比分析', 'thinking': '对比监督学习（依赖标签）、无监督学习（发现模式）与强化学习（目标导向的延迟奖励）。分析强化学习在处理动态环境、长期回报优化方面的不可替代性。', 'focus': '强调强化学习在需要连续决策、环境交互的场景中的优势，如游戏、机器人控制', 'word_count': 200, 'style': '对比结构清晰，使用术语准确，避免过度技术化'}
- {'id': 3, 'title': '关键应用领域与成功案例', 'direction': '实践验证', 'thinking': '列举强化学习在游戏（AlphaGo、AlphaStar）、机器人（灵巧操作、自主导航）、自动驾驶（决策规划）、推荐系统（用户互动优化）、大语言模型（RLHF）等领域的突破性应用。阐述其如何推动AI从感知走向决策。', 'focus': '用具体案例说明强化学习解决真实世界复杂问题的能力，强调其跨领域泛化性', 'word_count': 250, 'style': '案例驱动，数据支撑（如AlphaGo击败人类冠军），语言生动但不失客观'}
- {'id': 4, 'title': '强化学习在通用人工智能中的角色', 'direction': '战略价值', 'thinking': '讨论强化学习被认为是实现AGI的关键技术之一，因为它提供了智能体自主探索、适应环境、学习复杂技能的能力。对比当前深度学习在感知上的局限，强化学习在决策层的重要性。', 'focus': '论证强化学习赋予A

In [17]:
points = list(map(lambda prompt: point_agent.invoke({"prompt", json.dumps(prompt)}), prompts))
print(f"There are {len(points)} points, the first one is:")
print(points[0])

There are 5 points, the first one is:
强化学习是智能体通过与环境交互、依据奖惩信号学习最优策略的机器学习范式。其核心价值在于无需预标注数据，而是借助**试错探索**实现自主进化，与监督/无监督学习形成本质区别。该方法独特地适用于**序列决策**问题，通过累积奖励最大化的目标，模拟生物学习中的“奖惩”机制，使智能体在动态环境中逐步优化动作选择。这种从交互中自主习得策略的过程，既体现了对生物适应性的模仿，也为机器人控制、游戏博弈等领域提供了可拓展的解决方案。


In [22]:
abstract = syn_agent.invoke({"topic": topic, "points": points})
print("=== Final Abstract ===")
print(abstract)

=== Final Abstract ===
### 摘要
强化学习通过智能体与环境交互、借助奖惩信号试错探索，实现无需预标注数据的自主进化，与监督/无监督学习形成本质区别，特别适用于序列决策与动态环境。

- **核心机制**：以延迟奖励驱动长期累积回报最大化，模拟生物学习奖惩机制，能处理连续状态与动作空间，平衡即时与长期收益。
- **关键应用**：AlphaGo/AlphaStar攻克复杂博弈；机器人灵巧操作与自主导航成功率显著提升；自动驾驶轨迹规划降低干预率；推荐系统用户点击率提升12%～18%；RLHF将大语言模型升级为策略性生成。
- **战略价值**：赋予AI感知之外的行动-学习闭环，从静态模式识别迈向动态主动学习，是走向通用人工智能的关键支柱。
- **挑战与未来**：当前面临样本效率低、奖励设计困难、训练不稳定及安全风险。未来方向包括模型化强化学习、多智能体、逆强化学习及模仿学习，以提升效率与泛化能力。
